In [2]:
print("Glue Spark notebook is working.")
print(f"Spark version: {spark.version}")

Glue Spark notebook is working.
Spark version: 3.3.0-amzn-1


In [3]:
# ============================================================
# PART 1 — READ AND PROFILE
# ============================================================
# Purpose:
#   1. Read all 12 CSV files from the raw S3 layer.
#   2. Display the schema of every DataFrame.
#   3. Display 10 records from every table.
#   4. Calculate the number of records in every table.
#   5. Calculate NULL values for every column in customers.
#   6. Find duplicate customerid values.
#   7. Find duplicate orderid values.
#
# Assignment Reference:
#   Part 1 — Read and Profile
# ============================================================

from pyspark.sql import functions as F
import logging


# ============================================================
# 1. LOGGING CONFIGURATION
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("Ecommerce-Part1")

logger.info("=" * 70)
logger.info("STARTING PART 1 — READ AND PROFILE")
logger.info("=" * 70)


# ============================================================
# 2. S3 RAW DATA LOCATION
# ============================================================
# Each table is stored inside its own directory.
#
# Example:
# s3://aws-ecommerce-s3/ecommerce/raw/customers/customers.csv
#
# This folder structure allows each table to have its own
# independent Glue/Athena location.

RAW_PATH = "s3://aws-ecommerce-s3/ecommerce/raw"

logger.info(f"Raw data location: {RAW_PATH}")


# ============================================================
# 3. TABLE CONFIGURATION
# ============================================================
# Dictionary:
#     DataFrame name -> S3 CSV location
#
# We keep the DataFrames in a dictionary so that the same
# profiling logic can be applied to all tables.

table_paths = {
    "customers": f"{RAW_PATH}/customers/customers.csv",
    "categories": f"{RAW_PATH}/categories/categories.csv",
    "products": f"{RAW_PATH}/products/products.csv",
    "departments": f"{RAW_PATH}/departments/departments.csv",
    "employees": f"{RAW_PATH}/employees/employees.csv",
    "suppliers": f"{RAW_PATH}/suppliers/suppliers.csv",
    "orders": f"{RAW_PATH}/orders/orders.csv",
    "order_details": f"{RAW_PATH}/order_details/order_details.csv",
    "payments": f"{RAW_PATH}/payments/payments.csv",
    "product_suppliers": f"{RAW_PATH}/product_suppliers/product_suppliers.csv",
    "shippers": f"{RAW_PATH}/shippers/shippers.csv",
    "shipments": f"{RAW_PATH}/shipments/shipments.csv"
}


logger.info(f"Number of tables configured: {len(table_paths)}")


# ============================================================
# 4. READ ALL CSV FILES
# ============================================================
# header=True:
#     First row contains column names.
#
# inferSchema=True:
#     Spark attempts to automatically determine column data types.
#
# The resulting DataFrames are stored in the `tables` dictionary.

tables = {}

logger.info("Reading CSV files...")

for table_name, path in table_paths.items():

    logger.info(f"Reading table: {table_name}")
    logger.info(f"S3 path: {path}")

    try:
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .option("mode", "PERMISSIVE")
            .csv(path)
        )

        tables[table_name] = df

        logger.info(
            f"Successfully loaded {table_name} "
            f"with {len(df.columns)} columns"
        )

    except Exception as e:

        logger.error(
            f"Failed to read table {table_name}: {str(e)}"
        )

        raise


logger.info("All tables loaded successfully.")


# ============================================================
# 5. DISPLAY SCHEMA + 10 RECORDS FOR EVERY TABLE
# ============================================================
# Requirement:
#   Display the schema of every DataFrame.
#   Display 10 records from every table.
#
# truncate=False prevents long string values from being
# shortened in the displayed output.

logger.info("=" * 70)
logger.info("DISPLAYING SCHEMAS AND SAMPLE RECORDS")
logger.info("=" * 70)


for table_name, df in tables.items():

    print("\n")
    print("=" * 80)
    print(f"TABLE: {table_name.upper()}")
    print("=" * 80)

    # Display schema
    print("\n--- SCHEMA ---")
    df.printSchema()

    # Display first 10 records
    print("--- FIRST 10 RECORDS ---")
    df.show(10, truncate=False)


# ============================================================
# 6. RECORD COUNT FOR EVERY TABLE
# ============================================================
# Requirement:
#   Calculate the number of records in every table.
#
# We store the results in a small DataFrame so the counts can
# be viewed together instead of only printing individual values.

logger.info("=" * 70)
logger.info("CALCULATING RECORD COUNTS")
logger.info("=" * 70)


record_counts = []

for table_name, df in tables.items():

    count = df.count()

    record_counts.append(
        (table_name, count)
    )

    logger.info(
        f"{table_name}: {count:,} records"
    )


record_counts_df = (
    spark.createDataFrame(
        record_counts,
        ["table_name", "record_count"]
    )
    .orderBy("table_name")
)


print("\n")
print("=" * 80)
print("RECORD COUNT SUMMARY")
print("=" * 80)

record_counts_df.show(
    20,
    truncate=False
)


# ============================================================
# 7. NULL VALUES IN CUSTOMERS
# ============================================================
# Requirement:
#   Calculate the number of NULL values for every column
#   in the customers table.
#
# We create one row containing the NULL count for each column.

logger.info("=" * 70)
logger.info("CHECKING NULL VALUES IN CUSTOMERS")
logger.info("=" * 70)


customers = tables["customers"]


customer_null_counts = customers.select(
    *[
        F.sum(
            F.when(F.col(column).isNull(), 1)
             .otherwise(0)
        ).alias(column)
        for column in customers.columns
    ]
)


print("\n")
print("=" * 80)
print("CUSTOMERS — NULL VALUE COUNTS")
print("=" * 80)

customer_null_counts.show(
    truncate=False
)


# ============================================================
# 8. DUPLICATE CUSTOMER IDs
# ============================================================
# Requirement:
#   Find duplicate customerid values.
#
# groupBy(customerid) counts how many times each ID appears.
# count > 1 means that the ID occurs more than once.

logger.info("=" * 70)
logger.info("CHECKING DUPLICATE CUSTOMER IDs")
logger.info("=" * 70)


duplicate_customers = (
    customers
    .groupBy("customerid")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)


duplicate_customer_count = duplicate_customers.count()

logger.info(
    f"Number of duplicated customerid values: "
    f"{duplicate_customer_count:,}"
)


print("\n")
print("=" * 80)
print("DUPLICATE CUSTOMER IDs")
print("=" * 80)

if duplicate_customer_count > 0:

    duplicate_customers.show(
        50,
        truncate=False
    )

else:

    print("No duplicate customerid values found.")


# ============================================================
# 9. DUPLICATE ORDER IDs
# ============================================================
# Requirement:
#   Find duplicate orderid values.
#
# Again, an ID appearing more than once is considered a
# duplicate.

logger.info("=" * 70)
logger.info("CHECKING DUPLICATE ORDER IDs")
logger.info("=" * 70)


orders = tables["orders"]


duplicate_orders = (
    orders
    .groupBy("orderid")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)


duplicate_order_count = duplicate_orders.count()

logger.info(
    f"Number of duplicated orderid values: "
    f"{duplicate_order_count:,}"
)


print("\n")
print("=" * 80)
print("DUPLICATE ORDER IDs")
print("=" * 80)

if duplicate_order_count > 0:

    duplicate_orders.show(
        50,
        truncate=False
    )

else:

    print("No duplicate orderid values found.")


# ============================================================
# 10. PART 1 SUMMARY
# ============================================================

logger.info("=" * 70)
logger.info("PART 1 COMPLETED SUCCESSFULLY")
logger.info("=" * 70)

print("\n")
print("=" * 80)
print("PART 1 — PROFILING SUMMARY")
print("=" * 80)

print(f"Tables loaded       : {len(tables)}")
print(f"Duplicate customers : {duplicate_customer_count}")
print(f"Duplicate orders    : {duplicate_order_count}")

print("\nRecord counts:")
record_counts_df.show(20, truncate=False)

print("Part 1 completed.")



TABLE: CUSTOMERS

--- SCHEMA ---
root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- RegistrationDate: timestamp (nullable = true)

--- FIRST 10 RECORDS ---
+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|CustomerID|FirstName|LastName|Email                        |City      |Country       |RegistrationDate   |
+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|1         |Danielle |Johnson |danielle.johnson1@example.com|Giza      |Egypt         |2023-01-31 00:00:00|
|2         |Joshua   |Walker  |joshua.walker2@example.com   |Dubai     |Jordan        |2021-07-25 00:00:00|
|3         |Jill     |Rhodes  |jill.rhodes3@example.com     |Giza      |United Kingdom|2020-12-22 00:00:00|

In [4]:
# ============================================================
# PART 2 — DATA CLEANING
# ============================================================
# Purpose:
#   1. Standardize all column names to lowercase.
#   2. Remove leading/trailing spaces from string columns.
#   3. Convert customer emails to lowercase.
#   4. Convert order statuses to uppercase.
#   5. Remove duplicate customers based on customerid.
#   6. Remove duplicate orders based on orderid.
#   7. Remove records where the primary key is NULL.
#   8. Find customers with invalid email formats.
#   9. Find orders with invalid statuses.
#  10. Find products with NULL prices.
#  11. Find products with prices <= 0.
#
# Input:
#   `tables` dictionary created in Part 1.
#
# Output:
#   Cleaned DataFrames stored back in `tables`.
# ============================================================


# ============================================================
# 1. LOGGING
# ============================================================

logger.info("=" * 70)
logger.info("STARTING PART 2 — DATA CLEANING")
logger.info("=" * 70)


# ============================================================
# 2. STANDARDIZE ALL COLUMN NAMES TO LOWERCASE
# ============================================================
# Example:
#
#   CustomerID  -> customerid
#   FirstName   -> firstname
#   OrderDate   -> orderdate
#
# This makes column references consistent throughout the
# remaining pipeline.

logger.info("Standardizing all column names to lowercase...")


for table_name, df in tables.items():

    cleaned_columns = [
        column.lower()
        for column in df.columns
    ]

    df = df.toDF(*cleaned_columns)

    tables[table_name] = df

    logger.info(
        f"{table_name}: column names standardized"
    )


# ============================================================
# 3. REMOVE LEADING AND TRAILING SPACES
# ============================================================
# Only string columns are trimmed.
#
# Numeric/date columns are left unchanged.
#
# Example:
#   " Cairo " -> "Cairo"
#   "John "   -> "John"

logger.info("Removing leading and trailing spaces...")


for table_name, df in tables.items():

    string_columns = [
        field.name
        for field in df.schema.fields
        if field.dataType.simpleString() == "string"
    ]

    if string_columns:

        df = df.select(
            *[
                F.trim(F.col(column)).alias(column)
                if column in string_columns
                else F.col(column)
                for column in df.columns
            ]
        )

    tables[table_name] = df

    logger.info(
        f"{table_name}: trimmed {len(string_columns)} string columns"
    )


# ============================================================
# 4. CONVERT CUSTOMER EMAILS TO LOWERCASE
# ============================================================
# Email addresses are normalized to lowercase to avoid treating
# the same email with different capitalization as different
# values.
#
# Example:
#   John.Doe@EMAIL.COM -> john.doe@email.com

logger.info("Converting customer emails to lowercase...")


customers = tables["customers"]

if "email" in customers.columns:

    customers = customers.withColumn(
        "email",
        F.lower(F.col("email"))
    )

    logger.info("customers.email converted to lowercase")

else:

    logger.warning(
        "Column 'email' was not found in customers"
    )


tables["customers"] = customers


# ============================================================
# 5. CONVERT ORDER STATUSES TO UPPERCASE
# ============================================================
# The assignment defines the valid statuses as:
#
#   PENDING
#   SHIPPED
#   DELIVERED
#   CANCELLED
#
# We normalize the existing values to uppercase before
# validating them.

logger.info("Converting order statuses to uppercase...")


orders = tables["orders"]

if "status" in orders.columns:

    orders = orders.withColumn(
        "status",
        F.upper(F.col("status"))
    )

    logger.info("orders.status converted to uppercase")

else:

    logger.warning(
        "Column 'status' was not found in orders"
    )


tables["orders"] = orders


# ============================================================
# 6. REMOVE DUPLICATE CUSTOMERS
# ============================================================
# Duplicate customers are identified using customerid.
#
# dropDuplicates(["customerid"]) keeps one record for each
# customer ID.

logger.info("Removing duplicate customers based on customerid...")


customers = tables["customers"]

customers_before = customers.count()

customers = customers.dropDuplicates(
    ["customerid"]
)

customers_after = customers.count()

customers_removed = (
    customers_before - customers_after
)

logger.info(
    f"Customers before deduplication: {customers_before:,}"
)

logger.info(
    f"Customers after deduplication: {customers_after:,}"
)

logger.info(
    f"Duplicate customer records removed: {customers_removed:,}"
)


tables["customers"] = customers


# ============================================================
# 7. REMOVE DUPLICATE ORDERS
# ============================================================
# Duplicate orders are identified using orderid.

logger.info("Removing duplicate orders based on orderid...")


orders = tables["orders"]

orders_before = orders.count()

orders = orders.dropDuplicates(
    ["orderid"]
)

orders_after = orders.count()

orders_removed = (
    orders_before - orders_after
)

logger.info(
    f"Orders before deduplication: {orders_before:,}"
)

logger.info(
    f"Orders after deduplication: {orders_after:,}"
)

logger.info(
    f"Duplicate order records removed: {orders_removed:,}"
)


tables["orders"] = orders


# ============================================================
# 8. REMOVE RECORDS WITH NULL PRIMARY KEYS
# ============================================================
# The assignment requires removing records where the primary
# key is NULL.
#
# Primary keys used:
#
#   customers       -> customerid
#   categories      -> categoryid
#   products        -> productid
#   departments     -> departmentid
#   employees       -> employeeid
#   suppliers       -> supplierid
#   orders          -> orderid
#   order_details   -> orderdetailid
#   payments        -> paymentid
#   product_suppliers -> productsupplierid
#   shippers        -> shipperid
#   shipments       -> shipmentid
#
# If a particular dataset uses a different primary-key name,
# the code logs a warning instead of failing.

logger.info("Removing records with NULL primary keys...")


primary_keys = {
    "customers": "customerid",
    "categories": "categoryid",
    "products": "productid",
    "departments": "departmentid",
    "employees": "employeeid",
    "suppliers": "supplierid",
    "orders": "orderid",
    "order_details": "orderdetailid",
    "payments": "paymentid",
    "product_suppliers": "productsupplierid",
    "shippers": "shipperid",
    "shipments": "shipmentid"
}


null_pk_summary = []


for table_name, primary_key in primary_keys.items():

    df = tables[table_name]

    if primary_key not in df.columns:

        logger.warning(
            f"{table_name}: primary key '{primary_key}' "
            f"was not found"
        )

        continue

    before_count = df.count()

    null_pk_count = (
        df.filter(F.col(primary_key).isNull())
          .count()
    )

    df = df.filter(
        F.col(primary_key).isNotNull()
    )

    after_count = df.count()

    tables[table_name] = df

    null_pk_summary.append(
        (
            table_name,
            primary_key,
            null_pk_count,
            before_count,
            after_count
        )
    )

    logger.info(
        f"{table_name}: removed {null_pk_count:,} "
        f"records with NULL {primary_key}"
    )


# Display NULL primary-key summary

null_pk_df = spark.createDataFrame(
    null_pk_summary,
    [
        "table_name",
        "primary_key",
        "null_pk_records",
        "records_before",
        "records_after"
    ]
)

print("\n")
print("=" * 80)
print("NULL PRIMARY KEY CLEANING SUMMARY")
print("=" * 80)

null_pk_df.show(
    20,
    truncate=False
)


# ============================================================
# 9. FIND CUSTOMERS WITH INVALID EMAIL FORMATS
# ============================================================
# We do not delete these records here.
#
# This step identifies invalid email values so they can be
# reviewed.
#
# Basic validation:
#   something@something.something
#
# This is intentionally a simple data-quality check rather
# than a complete RFC email validator.

logger.info("Checking customers for invalid email formats...")


customers = tables["customers"]

email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"


invalid_customer_emails = (
    customers
    .filter(
        F.col("email").isNull()
        |
        (~F.col("email").rlike(email_pattern))
    )
)


invalid_email_count = invalid_customer_emails.count()


logger.info(
    f"Customers with invalid email formats: "
    f"{invalid_email_count:,}"
)


print("\n")
print("=" * 80)
print("INVALID CUSTOMER EMAILS")
print("=" * 80)

if invalid_email_count > 0:

    invalid_customer_emails.select(
        "customerid",
        "email"
    ).show(
        50,
        truncate=False
    )

else:

    print("No invalid customer email formats found.")


# ============================================================
# 10. FIND ORDERS WITH INVALID STATUSES
# ============================================================
# Valid values defined by the assignment:
#
#   PENDING
#   SHIPPED
#   DELIVERED
#   CANCELLED

logger.info("Checking orders for invalid statuses...")


valid_statuses = [
    "PENDING",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED"
]


orders = tables["orders"]


invalid_orders = (
    orders
    .filter(
        F.col("status").isNull()
        |
        (~F.col("status").isin(valid_statuses))
    )
)


invalid_order_count = invalid_orders.count()


logger.info(
    f"Orders with invalid statuses: "
    f"{invalid_order_count:,}"
)


print("\n")
print("=" * 80)
print("ORDERS WITH INVALID STATUS")
print("=" * 80)

if invalid_order_count > 0:

    invalid_orders.select(
        "orderid",
        "status"
    ).show(
        50,
        truncate=False
    )

else:

    print("No invalid order statuses found.")


# ============================================================
# 11. FIND PRODUCTS WITH NULL PRICES
# ============================================================
# Requirement:
#   Find products where price is NULL.
#
# We only identify the records here.
# The assignment does not say to delete them in Part 2.

logger.info("Checking products for NULL prices...")


products = tables["products"]


products_null_price = (
    products
    .filter(
        F.col("price").isNull()
    )
)


null_price_count = products_null_price.count()


logger.info(
    f"Products with NULL prices: "
    f"{null_price_count:,}"
)


print("\n")
print("=" * 80)
print("PRODUCTS WITH NULL PRICES")
print("=" * 80)

if null_price_count > 0:

    products_null_price.select(
        "productid",
        "price"
    ).show(
        50,
        truncate=False
    )

else:

    print("No products with NULL prices found.")


# ============================================================
# 12. FIND PRODUCTS WITH NON-POSITIVE PRICES
# ============================================================
# Requirement:
#   Find products with prices less than or equal to zero.

logger.info(
    "Checking products for prices less than or equal to zero..."
)


products_non_positive_price = (
    products
    .filter(
        F.col("price") <= 0
    )
)


non_positive_price_count = (
    products_non_positive_price.count()
)


logger.info(
    f"Products with price <= 0: "
    f"{non_positive_price_count:,}"
)


print("\n")
print("=" * 80)
print("PRODUCTS WITH PRICE <= 0")
print("=" * 80)

if non_positive_price_count > 0:

    products_non_positive_price.select(
        "productid",
        "price"
    ).show(
        50,
        truncate=False
    )

else:

    print("No products with price <= 0 found.")


# ============================================================
# 13. FINAL PART 2 SUMMARY
# ============================================================

logger.info("=" * 70)
logger.info("PART 2 COMPLETED SUCCESSFULLY")
logger.info("=" * 70)


print("\n")
print("=" * 80)
print("PART 2 — DATA CLEANING SUMMARY")
print("=" * 80)

print(f"Duplicate customers removed : {customers_removed:,}")
print(f"Duplicate orders removed    : {orders_removed:,}")
print(f"Invalid customer emails     : {invalid_email_count:,}")
print(f"Invalid order statuses      : {invalid_order_count:,}")
print(f"Products with NULL price    : {null_price_count:,}")
print(
    f"Products with price <= 0    : "
    f"{non_positive_price_count:,}"
)

print("\nPart 2 completed.")



NULL PRIMARY KEY CLEANING SUMMARY
+-------------+-------------+---------------+--------------+-------------+
|table_name   |primary_key  |null_pk_records|records_before|records_after|
+-------------+-------------+---------------+--------------+-------------+
|customers    |customerid   |0              |10000         |10000        |
|categories   |categoryid   |0              |20            |20           |
|products     |productid    |0              |1000          |1000         |
|departments  |departmentid |0              |10            |10           |
|employees    |employeeid   |0              |200           |200          |
|suppliers    |supplierid   |0              |100           |100          |
|orders       |orderid      |0              |50000         |50000        |
|order_details|orderdetailid|0              |100000        |100000       |
|payments     |paymentid    |0              |45000         |45000        |
|shippers     |shipperid    |0              |10            |10  

In [5]:
# ============================================================
# DEBUG — INSPECT ACTUAL ORDER STATUS VALUES
# ============================================================
# Purpose:
#   Investigate why a very large number of orders were detected
#   as having invalid statuses.
#
# This cell does NOT modify the orders DataFrame.
# It only displays the actual values and their frequencies.
# ============================================================

logger.info("=" * 70)
logger.info("INSPECTING ACTUAL ORDER STATUS VALUES")
logger.info("=" * 70)


orders = tables["orders"]


# ------------------------------------------------------------
# 1. Show every distinct status and its record count
# ------------------------------------------------------------
# This is the most important check.
#
# We want to see something like:
#
# status       count
# ----------   -----
# DELIVERED    ...
# SHIPPED      ...
# pending      ...
#
# If the dataset contains unexpected values, we will see them.

status_counts = (
    orders
    .groupBy("status")
    .count()
    .orderBy(F.col("count").desc())
)


print("\n")
print("=" * 80)
print("ACTUAL ORDER STATUS VALUES")
print("=" * 80)

status_counts.show(
    100,
    truncate=False
)


# ------------------------------------------------------------
# 2. Show the raw status values including their length
# ------------------------------------------------------------
# This helps detect hidden spaces or unexpected characters.
#
# Example:
#
# "SHIPPED"  -> length 7
# "SHIPPED " -> length 8
#
# We already applied trim() in Part 2, so this also helps
# confirm what remains after cleaning.

status_debug = (
    orders
    .select(
        "status",
        F.length("status").alias("status_length")
    )
    .distinct()
    .orderBy("status")
)


print("\n")
print("=" * 80)
print("STATUS VALUES + STRING LENGTH")
print("=" * 80)

status_debug.show(
    100,
    truncate=False
)


# ------------------------------------------------------------
# 3. Show the distribution using SQL-style grouping
# ------------------------------------------------------------

print("\n")
print("=" * 80)
print("STATUS DISTRIBUTION")
print("=" * 80)

status_counts.show(
    100,
    truncate=False
)


# ------------------------------------------------------------
# 4. Display sample orders for each status
# ------------------------------------------------------------
# This lets us see the actual records associated with the
# different status values.

print("\n")
print("=" * 80)
print("SAMPLE ORDERS WITH STATUS")
print("=" * 80)

orders.select(
    "orderid",
    "customerid",
    "orderdate",
    "status"
).show(
    50,
    truncate=False
)


logger.info("Order status inspection completed.")



ACTUAL ORDER STATUS VALUES
+----------+-----+
|status    |count|
+----------+-----+
|PENDING   |10044|
|SHIPPED   |10015|
|CANCELLED |9998 |
|DELIVERED |9975 |
|PROCESSING|9968 |
+----------+-----+



STATUS VALUES + STRING LENGTH
+----------+-------------+
|status    |status_length|
+----------+-------------+
|CANCELLED |9            |
|DELIVERED |9            |
|PENDING   |7            |
|PROCESSING|10           |
|SHIPPED   |7            |
+----------+-------------+



STATUS DISTRIBUTION
+----------+-----+
|status    |count|
+----------+-----+
|PENDING   |10044|
|SHIPPED   |10015|
|CANCELLED |9998 |
|DELIVERED |9975 |
|PROCESSING|9968 |
+----------+-----+



SAMPLE ORDERS WITH STATUS
+-------+----------+-------------------+----------+
|orderid|customerid|orderdate          |status    |
+-------+----------+-------------------+----------+
|1      |8870      |2024-05-03 00:00:00|CANCELLED |
|3      |8030      |2026-01-14 00:00:00|DELIVERED |
|5      |4623      |2025-05-26 00:00:00|D

In [7]:
# ============================================================
# PART 3 — DATA TYPE CONVERSION & STANDARDIZATION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import LongType, IntegerType, DecimalType, DateType

print("=" * 80)
print("PART 3 — DATA TYPE CONVERSION & STANDARDIZATION")
print("=" * 80)


# ============================================================
# 1. CONVERT ALL ID COLUMNS TO LONG
# ============================================================

print("\n[1/5] Converting ID columns to LONG...")

# Primary and foreign-key columns across the dataset
id_columns = {
    "customers": [
        "customerid"
    ],
    "categories": [
        "categoryid"
    ],
    "products": [
        "productid",
        "categoryid"
    ],
    "departments": [
        "departmentid"
    ],
    "employees": [
        "employeeid",
        "departmentid"
    ],
    "suppliers": [
        "supplierid"
    ],
    "orders": [
        "orderid",
        "customerid",
        "employeeid"
    ],
    "order_details": [
        "orderdetailid",
        "orderid",
        "productid"
    ],
    "payments": [
        "paymentid",
        "orderid"
    ],
    "product_suppliers": [
        "productsupplierid",
        "productid",
        "supplierid"
    ],
    "shippers": [
        "shipperid"
    ],
    "shipments": [
        "shipmentid",
        "orderid",
        "shipperid"
    ]
}

for table_name, columns in id_columns.items():

    df = tables[table_name]

    for column in columns:
        if column in df.columns:
            df = df.withColumn(
                column,
                F.col(column).cast(LongType())
            )

    tables[table_name] = df

    print(f"  ✓ {table_name}: ID columns converted")


# ============================================================
# 2. CONVERT PRICE COLUMNS TO DECIMAL(12,2)
# ============================================================

print("\n[2/5] Converting price columns to DECIMAL(12,2)...")

price_columns = {
    "products": ["price"],
    "order_details": ["unitprice"]
}

for table_name, columns in price_columns.items():

    df = tables[table_name]

    for column in columns:
        if column in df.columns:
            df = df.withColumn(
                column,
                F.col(column).cast(DecimalType(12, 2))
            )

    tables[table_name] = df

    print(f"  ✓ {table_name}: price columns converted")


# ============================================================
# 3. CONVERT ORDER DATE TO SPARK DATE
# ============================================================

print("\n[3/5] Converting orderdate to DATE...")

orders = tables["orders"]

orders = orders.withColumn(
    "orderdate",
    F.to_date(F.col("orderdate"))
)

tables["orders"] = orders

print("  ✓ orders.orderdate converted to DATE")


# ============================================================
# 4. CONVERT PAYMENT AMOUNT AND QUANTITY
# ============================================================

print("\n[4/5] Converting payment amounts and quantities...")

# Payment amount → DECIMAL(12,2)
payments = tables["payments"]

if "amount" in payments.columns:
    payments = payments.withColumn(
        "amount",
        F.col("amount").cast(DecimalType(12, 2))
    )

tables["payments"] = payments

# Quantity → INTEGER
order_details = tables["order_details"]

if "quantity" in order_details.columns:
    order_details = order_details.withColumn(
        "quantity",
        F.col("quantity").cast(IntegerType())
    )

tables["order_details"] = order_details

print("  ✓ payments.amount → DECIMAL(12,2)")
print("  ✓ order_details.quantity → INTEGER")


# ============================================================
# 5. EXTRACT DATE ATTRIBUTES FROM ORDERDATE
# ============================================================

print("\n[5/5] Extracting date attributes from orderdate...")

orders = tables["orders"]

orders = (
    orders
    .withColumn("year", F.year("orderdate"))
    .withColumn("month", F.month("orderdate"))
    .withColumn("quarter", F.quarter("orderdate"))
    .withColumn("day", F.dayofmonth("orderdate"))
    .withColumn("day_of_week", F.date_format("orderdate", "EEEE"))
)

tables["orders"] = orders

print("  ✓ year")
print("  ✓ month")
print("  ✓ quarter")
print("  ✓ day")
print("  ✓ day_of_week")


# ============================================================
# VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("PART 3 — VALIDATION")
print("=" * 80)

print("\nOrders schema:")
tables["orders"].printSchema()

print("\nOrder details schema:")
tables["order_details"].printSchema()

print("\nPayments schema:")
tables["payments"].printSchema()

print("\nSample orders with extracted date attributes:")

tables["orders"].select(
    "orderid",
    "orderdate",
    "year",
    "month",
    "quarter",
    "day",
    "day_of_week"
).show(10, truncate=False)

print("\n" + "=" * 80)
print("PART 3 COMPLETED")
print("=" * 80)

PART 3 — DATA TYPE CONVERSION & STANDARDIZATION

[1/5] Converting ID columns to LONG...
  ✓ customers: ID columns converted
  ✓ categories: ID columns converted
  ✓ products: ID columns converted
  ✓ departments: ID columns converted
  ✓ employees: ID columns converted
  ✓ suppliers: ID columns converted
  ✓ orders: ID columns converted
  ✓ order_details: ID columns converted
  ✓ payments: ID columns converted
  ✓ product_suppliers: ID columns converted
  ✓ shippers: ID columns converted
  ✓ shipments: ID columns converted

[2/5] Converting price columns to DECIMAL(12,2)...
  ✓ products: price columns converted
  ✓ order_details: price columns converted

[3/5] Converting orderdate to DATE...
  ✓ orders.orderdate converted to DATE

[4/5] Converting payment amounts and quantities...
  ✓ payments.amount → DECIMAL(12,2)
  ✓ order_details.quantity → INTEGER

[5/5] Extracting date attributes from orderdate...
  ✓ year
  ✓ month
  ✓ quarter
  ✓ day
  ✓ day_of_week

PART 3 — VALIDATION

Orders

In [8]:
# ============================================================
# PART 4 — DERIVED COLUMNS & BUSINESS LOGIC
# ============================================================

from pyspark.sql import functions as F

print("=" * 80)
print("PART 4 — DERIVED COLUMNS & BUSINESS LOGIC")
print("=" * 80)


# ============================================================
# 1. CALCULATE TOTAL AMOUNT FOR EACH ORDER DETAIL
# ============================================================
# Formula:
# total_amount = quantity * unitprice

print("\n[1/6] Calculating order detail total_amount...")

order_details = tables["order_details"]

order_details = order_details.withColumn(
    "total_amount",
    (F.col("quantity") * F.col("unitprice")).cast("decimal(12,2)")
)

tables["order_details"] = order_details

print("  ✓ Added order_details.total_amount")


# ============================================================
# 2. CREATE CUSTOMER FULL NAME
# ============================================================
# Combine first name and last name.
#
# concat_ws() is useful because it handles the separator
# cleanly between the two columns.

print("\n[2/6] Creating customer full_name...")

customers = tables["customers"]

customers = customers.withColumn(
    "full_name",
    F.concat_ws(
        " ",
        F.col("firstname"),
        F.col("lastname")
    )
)

tables["customers"] = customers

print("  ✓ Added customers.full_name")


# ============================================================
# 3. CREATE ORDER VALUE CATEGORY
# ============================================================
#
# HIGH   -> total amount >= 1000
# MEDIUM -> total amount between 500 and 999
# LOW    -> total amount < 500
#
# We first calculate the total value of every order from
# order_details, then classify each order.

print("\n[3/6] Creating order value categories...")

order_values = (
    order_details
    .groupBy("orderid")
    .agg(
        F.sum("total_amount").alias("order_total_amount")
    )
)

order_value_category = (
    order_values
    .withColumn(
        "order_value_category",
        F.when(
            F.col("order_total_amount") >= 1000,
            "HIGH"
        )
        .when(
            F.col("order_total_amount") >= 500,
            "MEDIUM"
        )
        .otherwise("LOW")
    )
)

# Join the category information back to orders
orders = tables["orders"]

orders = orders.join(
    order_value_category.select(
        "orderid",
        "order_value_category"
    ),
    on="orderid",
    how="left"
)

tables["orders"] = orders

print("  ✓ Added orders.order_value_category")


# ============================================================
# 4. CREATE CUSTOMER SEGMENT
# ============================================================
#
# Calculate total sales for every customer first.
#
# The assignment asks for customer segmentation based on
# total customer sales.
#
# Since the assignment does not explicitly define the
# thresholds for HIGH / MEDIUM / LOW customer segments,
# we use the same business thresholds:
#
# HIGH   -> >= 1000
# MEDIUM -> >= 500 and < 1000
# LOW    -> < 500
#
# This keeps the segmentation consistent with the
# order-value classification.

print("\n[4/6] Creating customer segments...")

customer_sales = (
    orders
    .join(
        order_details.select(
            "orderid",
            "total_amount"
        ),
        on="orderid",
        how="inner"
    )
    .groupBy("customerid")
    .agg(
        F.sum("total_amount").alias("total_customer_sales")
    )
)

customer_sales = (
    customer_sales
    .withColumn(
        "customer_segment",
        F.when(
            F.col("total_customer_sales") >= 1000,
            "HIGH"
        )
        .when(
            F.col("total_customer_sales") >= 500,
            "MEDIUM"
        )
        .otherwise("LOW")
    )
)

# Add the segment to customers
customers = tables["customers"]

customers = customers.join(
    customer_sales.select(
        "customerid",
        "total_customer_sales",
        "customer_segment"
    ),
    on="customerid",
    how="left"
)

tables["customers"] = customers

print("  ✓ Added customers.total_customer_sales")
print("  ✓ Added customers.customer_segment")


# ============================================================
# 5. TOTAL AMOUNT AND TOTAL QUANTITY PER ORDER
# ============================================================
#
# For every order:
#
# total_order_amount = SUM(total_amount)
# total_order_quantity = SUM(quantity)

print("\n[5/6] Calculating order totals...")

order_totals = (
    order_details
    .groupBy("orderid")
    .agg(
        F.sum("total_amount")
            .cast("decimal(12,2)")
            .alias("total_order_amount"),

        F.sum("quantity")
            .cast("integer")
            .alias("total_order_quantity")
    )
)

# Join the calculated metrics to orders
orders = tables["orders"]

orders = orders.join(
    order_totals,
    on="orderid",
    how="left"
)

tables["orders"] = orders

print("  ✓ Added orders.total_order_amount")
print("  ✓ Added orders.total_order_quantity")


# ============================================================
# 6. CALCULATE AVERAGE PRODUCT PRICE
# ============================================================
#
# Average price across the products table.

print("\n[6/6] Calculating average product price...")

products = tables["products"]

average_product_price = (
    products
    .agg(
        F.avg("price").cast("decimal(12,2)")
        .alias("average_product_price")
    )
)

print("  ✓ Average product price calculated")


# ============================================================
# VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("PART 4 — VALIDATION")
print("=" * 80)


# ------------------------------------------------------------
# Order details
# ------------------------------------------------------------

print("\nOrder Details:")
tables["order_details"].select(
    "orderdetailid",
    "orderid",
    "productid",
    "quantity",
    "unitprice",
    "total_amount"
).show(10, truncate=False)


# ------------------------------------------------------------
# Customers
# ------------------------------------------------------------

print("\nCustomers:")
tables["customers"].select(
    "customerid",
    "firstname",
    "lastname",
    "full_name",
    "total_customer_sales",
    "customer_segment"
).show(10, truncate=False)


# ------------------------------------------------------------
# Orders
# ------------------------------------------------------------

print("\nOrders:")
tables["orders"].select(
    "orderid",
    "customerid",
    "orderdate",
    "total_order_amount",
    "total_order_quantity",
    "order_value_category"
).show(10, truncate=False)


# ------------------------------------------------------------
# Order value category distribution
# ------------------------------------------------------------

print("\nOrder Value Category Distribution:")

tables["orders"].groupBy(
    "order_value_category"
).count().orderBy(
    F.col("count").desc()
).show()


# ------------------------------------------------------------
# Customer segment distribution
# ------------------------------------------------------------

print("\nCustomer Segment Distribution:")

tables["customers"].groupBy(
    "customer_segment"
).count().orderBy(
    F.col("count").desc()
).show()


# ------------------------------------------------------------
# Average product price
# ------------------------------------------------------------

print("\nAverage Product Price:")

average_product_price.show(truncate=False)


print("\n" + "=" * 80)
print("PART 4 COMPLETED")
print("=" * 80)

PART 4 — DERIVED COLUMNS & BUSINESS LOGIC

[1/6] Calculating order detail total_amount...
  ✓ Added order_details.total_amount

[2/6] Creating customer full_name...
  ✓ Added customers.full_name

[3/6] Creating order value categories...
  ✓ Added orders.order_value_category

[4/6] Creating customer segments...
  ✓ Added customers.total_customer_sales
  ✓ Added customers.customer_segment

[5/6] Calculating order totals...
  ✓ Added orders.total_order_amount
  ✓ Added orders.total_order_quantity

[6/6] Calculating average product price...
  ✓ Average product price calculated

PART 4 — VALIDATION

Order Details:
+-------------+-------+---------+--------+---------+------------+
|orderdetailid|orderid|productid|quantity|unitprice|total_amount|
+-------------+-------+---------+--------+---------+------------+
|1            |19370  |974      |5       |2877.48  |14387.40    |
|2            |49722  |403      |3       |1355.69  |4067.07     |
|3            |36676  |882      |9       |2249.53  |2

In [9]:
# ============================================================
# PART 5 — WINDOW FUNCTIONS & ADVANCED ANALYTICS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=" * 80)
print("PART 5 — WINDOW FUNCTIONS & ADVANCED ANALYTICS")
print("=" * 80)


# ============================================================
# 1. LATEST ORDER FOR EACH CUSTOMER
# ============================================================
# Partition by customerid so each customer gets their own window.
# Order by orderdate descending.
# row_number() = 1 gives the latest order.

print("\n[1/7] Finding latest order for each customer...")

orders = tables["orders"]

latest_order_window = (
    Window
    .partitionBy("customerid")
    .orderBy(
        F.col("orderdate").desc(),
        F.col("orderid").desc()
    )
)

latest_orders = (
    orders
    .withColumn(
        "row_num",
        F.row_number().over(latest_order_window)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print(f"  ✓ Latest order found for {latest_orders.count():,} customers")


# ============================================================
# 2. FIRST ORDER FOR EACH CUSTOMER
# ============================================================
# Same idea, but order by date ascending.

print("\n[2/7] Finding first order for each customer...")

first_order_window = (
    Window
    .partitionBy("customerid")
    .orderBy(
        F.col("orderdate").asc(),
        F.col("orderid").asc()
    )
)

first_orders = (
    orders
    .withColumn(
        "row_num",
        F.row_number().over(first_order_window)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print(f"  ✓ First order found for {first_orders.count():,} customers")


# ============================================================
# 3. RANK CUSTOMERS BY TOTAL SALES
# ============================================================
# We already calculated total_customer_sales in Part 4.
#
# rank() gives the same rank to customers with equal sales.
# Unlike row_number(), rank() does not force unique rankings.

print("\n[3/7] Ranking customers by total sales...")

customer_sales = (
    tables["customers"]
    .select(
        "customerid",
        "full_name",
        "total_customer_sales"
    )
    .filter(F.col("total_customer_sales").isNotNull())
)

customer_rank_window = (
    Window
    .orderBy(
        F.col("total_customer_sales").desc()
    )
)

ranked_customers = (
    customer_sales
    .withColumn(
        "sales_rank",
        F.rank().over(customer_rank_window)
    )
)

print("  ✓ Customers ranked by total sales")


# ============================================================
# 4. TOP 3 PRODUCTS IN EACH CATEGORY
# ============================================================
# First calculate product sales.
#
# Then rank products INSIDE each category.
#
# This is different from ranking all products globally.

print("\n[4/7] Finding top 3 products per category...")

products = tables["products"]
order_details = tables["order_details"]

product_sales = (
    order_details
    .groupBy("productid")
    .agg(
        F.sum("total_amount")
            .cast("decimal(18,2)")
            .alias("product_total_sales")
    )
)

product_category_sales = (
    products
    .select(
        "productid",
        "categoryid"
    )
    .join(
        product_sales,
        on="productid",
        how="left"
    )
)

category_product_window = (
    Window
    .partitionBy("categoryid")
    .orderBy(
        F.col("product_total_sales").desc_nulls_last(),
        F.col("productid").asc()
    )
)

ranked_category_products = (
    product_category_sales
    .withColumn(
        "product_rank",
        F.row_number().over(category_product_window)
    )
)

top_3_products_per_category = (
    ranked_category_products
    .filter(F.col("product_rank") <= 3)
)

print(
    f"  ✓ Found top 3 products per category "
    f"({top_3_products_per_category.count():,} records)"
)


# ============================================================
# 5. MOST EXPENSIVE PRODUCT IN EACH CATEGORY
# ============================================================
# Rank products within each category by price.
# row_number() = 1 gives the most expensive product.

print("\n[5/7] Finding most expensive product per category...")

expensive_product_window = (
    Window
    .partitionBy("categoryid")
    .orderBy(
        F.col("price").desc_nulls_last(),
        F.col("productid").asc()
    )
)

most_expensive_products = (
    products
    .withColumn(
        "price_rank",
        F.row_number().over(expensive_product_window)
    )
    .filter(F.col("price_rank") == 1)
    .drop("price_rank")
)

print(
    f"  ✓ Most expensive product found for "
    f"{most_expensive_products.count():,} categories"
)


# ============================================================
# 6. LATEST SHIPMENT FOR EACH ORDER
# ============================================================
# An order can potentially have multiple shipment records.
# We partition by orderid and select the latest shipment.

print("\n[6/7] Finding latest shipment for each order...")

shipments = tables["shipments"]

# Detect a suitable shipment date column.
shipment_date_candidates = [
    "shipmentdate",
    "shipdate",
    "shippeddate",
    "shipment_date",
    "ship_date"
]

shipment_date_column = next(
    (
        column
        for column in shipment_date_candidates
        if column in shipments.columns
    ),
    None
)

if shipment_date_column is not None:

    latest_shipment_window = (
        Window
        .partitionBy("orderid")
        .orderBy(
            F.col(shipment_date_column).desc_nulls_last(),
            F.col("shipmentid").desc()
        )
    )

    latest_shipments = (
        shipments
        .withColumn(
            "row_num",
            F.row_number().over(latest_shipment_window)
        )
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    print(
        f"  ✓ Latest shipment found for "
        f"{latest_shipments.count():,} orders"
    )

else:

    print(
        "  ! No recognized shipment date column found."
    )

    print(
        "  Available shipment columns:",
        shipments.columns
    )

    latest_shipments = shipments


# ============================================================
# 7. LATEST ORDER OVERALL
# ============================================================
# Find the single most recent order in the entire dataset.

print("\n[7/7] Finding latest order overall...")

latest_order = (
    orders
    .orderBy(
        F.col("orderdate").desc_nulls_last(),
        F.col("orderid").desc()
    )
    .limit(1)
)

print("  ✓ Latest order identified")


# ============================================================
# VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("PART 5 — VALIDATION")
print("=" * 80)


# ------------------------------------------------------------
# Latest order per customer
# ------------------------------------------------------------

print("\n1. Latest order per customer:")

latest_orders.select(
    "customerid",
    "orderid",
    "orderdate",
    "status"
).show(10, truncate=False)


# ------------------------------------------------------------
# First order per customer
# ------------------------------------------------------------

print("\n2. First order per customer:")

first_orders.select(
    "customerid",
    "orderid",
    "orderdate",
    "status"
).show(10, truncate=False)


# ------------------------------------------------------------
# Customer ranking
# ------------------------------------------------------------

print("\n3. Customer ranking by total sales:")

ranked_customers.select(
    "customerid",
    "full_name",
    "total_customer_sales",
    "sales_rank"
).orderBy(
    "sales_rank"
).show(10, truncate=False)


# ------------------------------------------------------------
# Top 3 products per category
# ------------------------------------------------------------

print("\n4. Top 3 products per category:")

top_3_products_per_category.select(
    "categoryid",
    "productid",
    "product_total_sales",
    "product_rank"
).orderBy(
    "categoryid",
    "product_rank"
).show(20, truncate=False)


# ------------------------------------------------------------
# Most expensive product per category
# ------------------------------------------------------------

print("\n5. Most expensive product per category:")

most_expensive_products.select(
    "categoryid",
    "productid",
    "price"
).orderBy(
    "categoryid"
).show(20, truncate=False)


# ------------------------------------------------------------
# Latest shipment
# ------------------------------------------------------------

print("\n6. Latest shipments:")

latest_shipments.show(10, truncate=False)


# ------------------------------------------------------------
# Latest order overall
# ------------------------------------------------------------

print("\n7. Latest order overall:")

latest_order.select(
    "orderid",
    "customerid",
    "orderdate",
    "status"
).show(truncate=False)


print("\n" + "=" * 80)
print("PART 5 COMPLETED")
print("=" * 80)

PART 5 — WINDOW FUNCTIONS & ADVANCED ANALYTICS

[1/7] Finding latest order for each customer...
  ✓ Latest order found for 9,931 customers

[2/7] Finding first order for each customer...
  ✓ First order found for 9,931 customers

[3/7] Ranking customers by total sales...
  ✓ Customers ranked by total sales

[4/7] Finding top 3 products per category...
  ✓ Found top 3 products per category (60 records)

[5/7] Finding most expensive product per category...
  ✓ Most expensive product found for 20 categories

[6/7] Finding latest shipment for each order...
  ✓ Latest shipment found for 27,459 orders

[7/7] Finding latest order overall...
  ✓ Latest order identified

PART 5 — VALIDATION

1. Latest order per customer:
+----------+-------+----------+----------+
|customerid|orderid|orderdate |status    |
+----------+-------+----------+----------+
|3         |17011  |2026-04-01|PENDING   |
|4         |21695  |2025-10-15|PROCESSING|
|6         |19312  |2026-07-03|PENDING   |
|9         |2861   |

In [11]:
# ============================================================
# PART 6 — JOINS & COMBINED DATASETS
# ============================================================

from pyspark.sql import functions as F

print("=" * 80)
print("PART 6 — JOINS & COMBINED DATASETS")
print("=" * 80)


# ============================================================
# LOAD TABLES
# ============================================================

customers = tables["customers"]
categories = tables["categories"]
products = tables["products"]
suppliers = tables["suppliers"]
orders = tables["orders"]
order_details = tables["order_details"]
payments = tables["payments"]
product_suppliers = tables["product_suppliers"]
shipments = tables["shipments"]
shippers = tables["shippers"]


# ============================================================
# 1. ORDERS + CUSTOMERS
# ============================================================
# Connect every order with the customer who placed it.
#
# Join key:
# orders.customerid = customers.customerid

print("\n[1/6] Joining orders with customers...")

orders_customers = (
    orders.alias("o")
    .join(
        customers.alias("c"),
        F.col("o.customerid") == F.col("c.customerid"),
        "left"
    )
    .select(
        F.col("o.orderid"),
        F.col("o.customerid"),
        F.col("c.full_name"),
        F.col("c.email"),
        F.col("o.orderdate"),
        F.col("o.status"),
        F.col("o.total_order_amount"),
        F.col("o.total_order_quantity"),
        F.col("o.order_value_category")
    )
)

print(
    f"  ✓ orders_customers created: "
    f"{orders_customers.count():,} records"
)


# ============================================================
# 2. ORDERS + ORDER_DETAILS + PRODUCTS
#    → SALES DATASET
# ============================================================
#
# This is the main transactional sales dataset.
#
# Relationship:
#
# orders
#    |
#    | orderid
#    ↓
# order_details
#    |
#    | productid
#    ↓
# products

print("\n[2/6] Building sales dataset...")

sales = (
    orders.alias("o")
    .join(
        order_details.alias("od"),
        F.col("o.orderid") == F.col("od.orderid"),
        "inner"
    )
    .join(
        products.alias("p"),
        F.col("od.productid") == F.col("p.productid"),
        "left"
    )
    .select(
        F.col("o.orderid"),
        F.col("o.customerid"),
        F.col("od.orderdetailid"),
        F.col("od.productid"),
        F.col("p.categoryid"),
        F.col("o.orderdate"),
        F.col("od.quantity"),
        F.col("od.unitprice"),
        F.col("od.total_amount"),
        F.col("o.status"),
        F.col("o.order_value_category")
    )
)

print(
    f"  ✓ sales dataset created: "
    f"{sales.count():,} records"
)


# ============================================================
# 3. PRODUCTS + CATEGORIES
# ============================================================
# Connect each product to its category.

print("\n[3/6] Joining products with categories...")

products_categories = (
    products.alias("p")
    .join(
        categories.alias("c"),
        F.col("p.categoryid") == F.col("c.categoryid"),
        "left"
    )
    .select(
        F.col("p.productid"),
        F.col("p.categoryid"),
        F.col("c.categoryname"),
        F.col("p.productname"),
        F.col("p.price")
    )
)

print(
    f"  ✓ products_categories created: "
    f"{products_categories.count():,} records"
)


# ============================================================
# 4. PRODUCTS + PRODUCT_SUPPLIERS + SUPPLIERS
# ============================================================
#
# A product can have supplier relationships stored in
# product_suppliers.
#
# product_suppliers connects:
#
# productid → products
# supplierid → suppliers

print("\n[4/6] Building product-supplier dataset...")

products_suppliers = (
    products.alias("p")
    .join(
        product_suppliers.alias("ps"),
        F.col("p.productid") == F.col("ps.productid"),
        "left"
    )
    .join(
        suppliers.alias("s"),
        F.col("ps.supplierid") == F.col("s.supplierid"),
        "left"
    )
    .select(
        F.col("p.productid"),
        F.col("p.productname"),
        F.col("p.categoryid"),
        F.col("ps.supplierid"),
        F.col("s.suppliername")
    )
)

print(
    f"  ✓ products_suppliers created: "
    f"{products_suppliers.count():,} records"
)


# ============================================================
# 5. ORDERS + PAYMENTS
# ============================================================
#
# Connect payments to their corresponding orders.
#
# Join key:
# orders.orderid = payments.orderid

print("\n[5/6] Joining orders with payments...")

orders_payments = (
    orders.alias("o")
    .join(
        payments.alias("p"),
        F.col("o.orderid") == F.col("p.orderid"),
        "left"
    )
    .select(
        F.col("o.orderid"),
        F.col("o.customerid"),
        F.col("o.orderdate"),
        F.col("o.status"),
        F.col("o.total_order_amount"),
        F.col("p.paymentid"),
        F.col("p.amount").alias("payment_amount")
    )
)

print(
    f"  ✓ orders_payments created: "
    f"{orders_payments.count():,} records"
)



# ============================================================
# 6. ORDERS + SHIPMENTS + SHIPPERS
# ============================================================

print("\n[6/6] Building order-shipment dataset...")

orders_shipments = (
    orders.alias("o")
    .join(
        shipments.alias("sh"),
        F.col("o.orderid") == F.col("sh.orderid"),
        "left"
    )
    .join(
        shippers.alias("sp"),
        F.col("sh.shipperid") == F.col("sp.shipperid"),
        "left"
    )
    .select(
        F.col("o.orderid"),
        F.col("o.customerid"),
        F.col("o.orderdate"),
        F.col("o.status"),
        F.col("sh.shipmentid"),
        F.col("sh.shipperid"),
        F.col("sp.companyname").alias("shipper_name")
    )
)

print(
    f"  ✓ orders_shipments created: "
    f"{orders_shipments.count():,} records"
)

# ============================================================
# VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("PART 6 — VALIDATION")
print("=" * 80)


# ------------------------------------------------------------
# 1. Orders + Customers
# ------------------------------------------------------------

print("\n1. ORDERS + CUSTOMERS")

orders_customers.show(10, truncate=False)


# ------------------------------------------------------------
# 2. Sales Dataset
# ------------------------------------------------------------

print("\n2. SALES DATASET")

sales.show(10, truncate=False)


# ------------------------------------------------------------
# 3. Products + Categories
# ------------------------------------------------------------

print("\n3. PRODUCTS + CATEGORIES")

products_categories.show(10, truncate=False)


# ------------------------------------------------------------
# 4. Products + Suppliers
# ------------------------------------------------------------

print("\n4. PRODUCTS + SUPPLIERS")

products_suppliers.show(10, truncate=False)


# ------------------------------------------------------------
# 5. Orders + Payments
# ------------------------------------------------------------

print("\n5. ORDERS + PAYMENTS")

orders_payments.show(10, truncate=False)


# ------------------------------------------------------------
# 6. Orders + Shipments + Shippers
# ------------------------------------------------------------

print("\n6. ORDERS + SHIPMENTS + SHIPPERS")

orders_shipments.show(10, truncate=False)


# ============================================================
# SAVE DATASETS IN MEMORY FOR NEXT PARTS
# ============================================================

print("\nSaving combined datasets...")

joined_tables = {
    "orders_customers": orders_customers,
    "sales": sales,
    "products_categories": products_categories,
    "products_suppliers": products_suppliers,
    "orders_payments": orders_payments,
    "orders_shipments": orders_shipments
}

print("  ✓ All six joined datasets are ready")


print("\n" + "=" * 80)
print("PART 6 COMPLETED")
print("=" * 80)

PART 6 — JOINS & COMBINED DATASETS

[1/6] Joining orders with customers...
  ✓ orders_customers created: 50,000 records

[2/6] Building sales dataset...
  ✓ sales dataset created: 100,000 records

[3/6] Joining products with categories...
  ✓ products_categories created: 1,000 records

[4/6] Building product-supplier dataset...
  ✓ products_suppliers created: 2,027 records

[5/6] Joining orders with payments...
  ✓ orders_payments created: 65,333 records

[6/6] Building order-shipment dataset...
  ✓ orders_shipments created: 62,541 records

PART 6 — VALIDATION

1. ORDERS + CUSTOMERS
+-------+----------+---------------+-------------------------------+----------+----------+------------------+--------------------+--------------------+
|orderid|customerid|full_name      |email                          |orderdate |status    |total_order_amount|total_order_quantity|order_value_category|
+-------+----------+---------------+-------------------------------+----------+----------+----------------

In [12]:
# ============================================================
# PART 7 — AGGREGATIONS
# ============================================================

from pyspark.sql import functions as F
import logging

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------

logger = logging.getLogger("Part7_Aggregations")
logger.setLevel(logging.INFO)

logger.info("Starting Part 7 — Aggregations")


# ------------------------------------------------------------
# 1. Overall sales metrics
# ------------------------------------------------------------
# Calculate:
# - Total sales
# - Total number of orders
# - Total quantity sold
# - Average order value
#
# We use the sales dataset created in Part 6.
# Each row represents an order-detail/product sale.
# ------------------------------------------------------------

overall_metrics = (
    sales
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("orderid").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.avg("total_amount").alias("average_order_value")
    )
)

print("=== Overall Sales Metrics ===")
overall_metrics.show(truncate=False)


# ------------------------------------------------------------
# 2. Sales by customer
# ------------------------------------------------------------
# Calculate total sales and number of orders for each customer.
# ------------------------------------------------------------

customer_sales = (
    sales
    .groupBy("customerid")
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("orderid").alias("total_orders"),
        F.sum("quantity").alias("total_quantity")
    )
    .orderBy(F.col("total_sales").desc())
)

print("=== Sales by Customer ===")
customer_sales.show(10, truncate=False)


# ------------------------------------------------------------
# 3. Sales by product
# ------------------------------------------------------------
# Calculate:
# - Total sales
# - Total quantity sold
# - Number of orders
# for each product.
# ------------------------------------------------------------

product_sales = (
    sales
    .groupBy("productid")
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("orderid").alias("total_orders")
    )
    .orderBy(F.col("total_sales").desc())
)

print("=== Sales by Product ===")
product_sales.show(10, truncate=False)


# ------------------------------------------------------------
# 4. Sales by category
# ------------------------------------------------------------
# Calculate sales metrics for every product category.
# ------------------------------------------------------------

category_sales = (
    sales
    .groupBy("categoryid")
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("orderid").alias("total_orders")
    )
    .orderBy(F.col("total_sales").desc())
)

print("=== Sales by Category ===")
category_sales.show(10, truncate=False)


# ------------------------------------------------------------
# 5. Sales by country
# ------------------------------------------------------------
# The country information comes from the customers table,
# so use the orders_customers dataset from Part 6.
#
# Join it with sales to connect customer location with sales.
# ------------------------------------------------------------

sales_by_country = (
    sales
    .join(
        customers.select("customerid", "country"),
        on="customerid",
        how="left"
    )
    .groupBy("country")
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("orderid").alias("total_orders"),
        F.sum("quantity").alias("total_quantity")
    )
    .orderBy(F.col("total_sales").desc())
)

print("=== Sales by Country ===")
sales_by_country.show(10, truncate=False)


# ------------------------------------------------------------
# 6. Sales by city
# ------------------------------------------------------------

sales_by_city = (
    sales
    .join(
        customers.select("customerid", "city"),
        on="customerid",
        how="left"
    )
    .groupBy("city")
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("orderid").alias("total_orders"),
        F.sum("quantity").alias("total_quantity")
    )
    .orderBy(F.col("total_sales").desc())
)

print("=== Sales by City ===")
sales_by_city.show(10, truncate=False)


# ------------------------------------------------------------
# 7. Sales by month
# ------------------------------------------------------------
# orderdate was converted to Spark date in Part 3.
#
# Group by year and month so that January from different
# years does not get mixed together.
# ------------------------------------------------------------

sales_by_month = (
    sales
    .filter(F.col("orderdate").isNotNull())
    .groupBy(
        F.year("orderdate").alias("year"),
        F.month("orderdate").alias("month")
    )
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("orderid").alias("total_orders"),
        F.sum("quantity").alias("total_quantity")
    )
    .orderBy("year", "month")
)

print("=== Sales by Month ===")
sales_by_month.show(20, truncate=False)


# ------------------------------------------------------------
# 8. Sales by year
# ------------------------------------------------------------

sales_by_year = (
    sales
    .filter(F.col("orderdate").isNotNull())
    .groupBy(
        F.year("orderdate").alias("year")
    )
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("orderid").alias("total_orders"),
        F.sum("quantity").alias("total_quantity")
    )
    .orderBy("year")
)

print("=== Sales by Year ===")
sales_by_year.show(20, truncate=False)


# ------------------------------------------------------------
# 9. Sales by order status
# ------------------------------------------------------------
# Use the status from the orders table.
# ------------------------------------------------------------

sales_by_status = (
    sales
    .groupBy("status")
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("orderid").alias("total_orders"),
        F.sum("quantity").alias("total_quantity")
    )
    .orderBy(F.col("total_sales").desc())
)

print("=== Sales by Order Status ===")
sales_by_status.show(20, truncate=False)


# ------------------------------------------------------------
# 10. Orders per customer
# ------------------------------------------------------------
# Count the number of distinct orders for every customer.
# ------------------------------------------------------------

orders_per_customer = (
    orders
    .groupBy("customerid")
    .agg(
        F.countDistinct("orderid").alias("order_count")
    )
    .orderBy(F.col("order_count").desc())
)

print("=== Orders per Customer ===")
orders_per_customer.show(10, truncate=False)


# ------------------------------------------------------------
# Store Part 7 outputs
# ------------------------------------------------------------

part7_results = {
    "overall_metrics": overall_metrics,
    "customer_sales": customer_sales,
    "product_sales": product_sales,
    "category_sales": category_sales,
    "sales_by_country": sales_by_country,
    "sales_by_city": sales_by_city,
    "sales_by_month": sales_by_month,
    "sales_by_year": sales_by_year,
    "sales_by_status": sales_by_status,
    "orders_per_customer": orders_per_customer
}


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\n=== Part 7 Validation ===")

for name, df in part7_results.items():
    print(f"{name}: {df.count()} rows")

logger.info("Part 7 completed successfully.")

=== Overall Sales Metrics ===
+------------+------------+--------------+-------------------+
|total_sales |total_orders|total_quantity|average_order_value|
+------------+------------+--------------+-------------------+
|840507631.75|43136       |549473        |8405.076318        |
+------------+------------+--------------+-------------------+

=== Sales by Customer ===
+----------+-----------+------------+--------------+
|customerid|total_sales|total_orders|total_quantity|
+----------+-----------+------------+--------------+
|6772      |368984.90  |12          |211           |
|245       |338329.93  |9           |192           |
|9895      |330803.47  |7           |198           |
|5446      |321600.74  |9           |184           |
|5688      |319630.49  |11          |169           |
|4971      |314924.38  |11          |182           |
|7013      |311641.81  |10          |196           |
|8502      |310361.58  |12          |199           |
|3556      |309252.08  |14          |178     

In [13]:
# ============================================================
# PART 8 — ADVANCED ANALYTICS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import logging

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------

logger = logging.getLogger("Part8_AdvancedAnalytics")
logger.setLevel(logging.INFO)

logger.info("Starting Part 8 — Advanced Analytics")


# ============================================================
# 1. TOP 10 CUSTOMERS BY TOTAL SALES
# ============================================================
# Join customer information with the customer sales
# calculated in Part 7.
# ------------------------------------------------------------

top_10_customers = (
    customer_sales
    .join(
        customers.select(
            "customerid",
            "firstname",
            "lastname",
            "email",
            "country",
            "city"
        ),
        on="customerid",
        how="left"
    )
    .select(
        "customerid",
        "firstname",
        "lastname",
        "email",
        "country",
        "city",
        "total_sales",
        "total_orders",
        "total_quantity"
    )
    .orderBy(F.col("total_sales").desc())
    .limit(10)
)

print("=== Top 10 Customers by Total Sales ===")
top_10_customers.show(10, truncate=False)


# ============================================================
# 2. TOP 10 PRODUCTS BY TOTAL SALES
# ============================================================

top_10_products = (
    product_sales
    .join(
        products.select(
            "productid",
            "productname",
            "categoryid",
            "price"
        ),
        on="productid",
        how="left"
    )
    .select(
        "productid",
        "productname",
        "categoryid",
        "price",
        "total_sales",
        "total_quantity",
        "total_orders"
    )
    .orderBy(F.col("total_sales").desc())
    .limit(10)
)

print("=== Top 10 Products by Total Sales ===")
top_10_products.show(10, truncate=False)


# ============================================================
# 3. TOP 5 CATEGORIES BY TOTAL SALES
# ============================================================

top_5_categories = (
    category_sales
    .join(
        categories.select(
            "categoryid",
            "categoryname"
        ),
        on="categoryid",
        how="left"
    )
    .select(
        "categoryid",
        "categoryname",
        "total_sales",
        "total_quantity",
        "total_orders"
    )
    .orderBy(F.col("total_sales").desc())
    .limit(5)
)

print("=== Top 5 Categories by Total Sales ===")
top_5_categories.show(5, truncate=False)


# ============================================================
# 4. CUSTOMERS WHO NEVER PLACED AN ORDER
# ============================================================
# LEFT ANTI JOIN returns customers that have no matching
# customerid in the orders table.
# ------------------------------------------------------------

customers_never_ordered = (
    customers
    .join(
        orders.select("customerid").distinct(),
        on="customerid",
        how="left_anti"
    )
)

print("=== Customers Who Never Placed an Order ===")
customers_never_ordered.show(20, truncate=False)

print(
    "Customers never ordered:",
    customers_never_ordered.count()
)


# ============================================================
# 5. PRODUCTS THAT WERE NEVER ORDERED
# ============================================================
# Compare products against order_details.
# ------------------------------------------------------------

products_never_ordered = (
    products
    .join(
        order_details.select("productid").distinct(),
        on="productid",
        how="left_anti"
    )
)

print("=== Products That Were Never Ordered ===")
products_never_ordered.show(20, truncate=False)

print(
    "Products never ordered:",
    products_never_ordered.count()
)


# ============================================================
# 6. CUSTOMERS WITH MORE THAN 10 ORDERS
# ============================================================

customers_more_than_10_orders = (
    orders_per_customer
    .filter(F.col("order_count") > 10)
    .join(
        customers.select(
            "customerid",
            "firstname",
            "lastname",
            "email"
        ),
        on="customerid",
        how="left"
    )
    .select(
        "customerid",
        "firstname",
        "lastname",
        "email",
        "order_count"
    )
    .orderBy(F.col("order_count").desc())
)

print("=== Customers With More Than 10 Orders ===")
customers_more_than_10_orders.show(20, truncate=False)

print(
    "Customers with >10 orders:",
    customers_more_than_10_orders.count()
)


# ============================================================
# 7. PAYMENT MISMATCH
# ============================================================
# Compare:
#
#   Order total
#       VS
#   Total amount paid
#
# We aggregate payments because an order may have multiple
# payment records.
# ------------------------------------------------------------

order_totals = (
    order_details
    .groupBy("orderid")
    .agg(
        F.sum("total_amount")
        .cast("decimal(12,2)")
        .alias("order_total")
    )
)

payment_totals = (
    payments
    .groupBy("orderid")
    .agg(
        F.sum("amount")
        .cast("decimal(12,2)")
        .alias("paid_total")
    )
)

payment_mismatch = (
    order_totals
    .join(
        payment_totals,
        on="orderid",
        how="left"
    )
    .withColumn(
        "paid_total",
        F.coalesce(
            F.col("paid_total"),
            F.lit(0).cast("decimal(12,2)")
        )
    )
    .withColumn(
        "difference",
        (F.col("order_total") - F.col("paid_total"))
        .cast("decimal(12,2)")
    )
    .filter(
        F.abs(F.col("difference").cast("double")) > 0.01
    )
    .orderBy(
        F.abs(F.col("difference").cast("double")).desc()
    )
)

print("=== Payment Mismatches ===")
payment_mismatch.show(20, truncate=False)

print(
    "Payment mismatch records:",
    payment_mismatch.count()
)


# ============================================================
# 8. ORDERS WITH NO SHIPMENT
# ============================================================
# Find orders that do not have a matching shipment record.
# ------------------------------------------------------------

orders_no_shipment = (
    orders
    .join(
        shipments.select("orderid").distinct(),
        on="orderid",
        how="left_anti"
    )
)

print("=== Orders With No Shipment ===")
orders_no_shipment.show(20, truncate=False)

print(
    "Orders with no shipment:",
    orders_no_shipment.count()
)


# ============================================================
# 9. SHIPMENTS WITH NO MATCHING ORDER
# ============================================================
# Find shipment records whose orderid does not exist
# in the orders table.
# ------------------------------------------------------------

shipments_no_matching_order = (
    shipments
    .join(
        orders.select("orderid").distinct(),
        on="orderid",
        how="left_anti"
    )
)

print("=== Shipments With No Matching Order ===")
shipments_no_matching_order.show(20, truncate=False)

print(
    "Shipments without matching order:",
    shipments_no_matching_order.count()
)


# ============================================================
# 10. STORE PART 8 RESULTS
# ============================================================

part8_results = {
    "top_10_customers": top_10_customers,
    "top_10_products": top_10_products,
    "top_5_categories": top_5_categories,
    "customers_never_ordered": customers_never_ordered,
    "products_never_ordered": products_never_ordered,
    "customers_more_than_10_orders": customers_more_than_10_orders,
    "payment_mismatch": payment_mismatch,
    "orders_no_shipment": orders_no_shipment,
    "shipments_no_matching_order": shipments_no_matching_order
}


# ============================================================
# VALIDATION
# ============================================================

print("\n=== Part 8 Validation ===")

for name, df in part8_results.items():
    print(f"{name}: {df.count()} rows")


logger.info("Part 8 completed successfully.")

=== Top 10 Customers by Total Sales ===
+----------+---------+---------+--------------------------------+--------------+---------+-----------+------------+--------------+
|customerid|firstname|lastname |email                           |country       |city     |total_sales|total_orders|total_quantity|
+----------+---------+---------+--------------------------------+--------------+---------+-----------+------------+--------------+
|6772      |Angela   |Stewart  |angela.stewart6772@example.com  |United Kingdom|Abu Dhabi|368984.90  |12          |211           |
|245       |Amy      |Morrison |amy.morrison245@example.com     |Jordan        |Berlin   |338329.93  |9           |192           |
|9895      |Benjamin |Holder   |benjamin.holder9895@example.com |France        |Jeddah   |330803.47  |7           |198           |
|5446      |Lisa     |Hogan    |lisa.hogan5446@example.com      |Jordan        |Giza     |321600.74  |9           |184           |
|5688      |Jeffrey  |Ramirez  |jeffrey.ram

In [14]:
# ============================================================
# INVESTIGATION — PAYMENT MISMATCHES & MISSING SHIPMENTS
# ============================================================

from pyspark.sql import functions as F

# ============================================================
# A. PAYMENT MISMATCH INVESTIGATION
# ============================================================

print("=" * 70)
print("PAYMENT MISMATCH INVESTIGATION")
print("=" * 70)


# ------------------------------------------------------------
# A1. Basic payment coverage
# ------------------------------------------------------------

print("\n=== Payment Coverage ===")

payment_coverage = (
    orders
    .select("orderid")
    .join(
        payments.select("orderid").distinct(),
        on="orderid",
        how="left"
    )
    .withColumn(
        "has_payment",
        F.when(F.col("orderid").isNotNull(), F.lit(1)).otherwise(0)
    )
)

# Better explicit check using left anti
orders_without_payment = (
    orders
    .join(
        payments.select("orderid").distinct(),
        on="orderid",
        how="left_anti"
    )
)

print("Orders without any payment:", orders_without_payment.count())


# ------------------------------------------------------------
# A2. Compare order total vs payment total
# ------------------------------------------------------------

order_totals = (
    order_details
    .groupBy("orderid")
    .agg(
        F.sum("total_amount")
        .cast("decimal(12,2)")
        .alias("order_total")
    )
)

payment_totals = (
    payments
    .groupBy("orderid")
    .agg(
        F.sum("amount")
        .cast("decimal(12,2)")
        .alias("paid_total")
    )
)

payment_analysis = (
    orders
    .select("orderid", "customerid", "orderdate", "status")
    .join(order_totals, "orderid", "left")
    .join(payment_totals, "orderid", "left")
    .withColumn(
        "paid_total",
        F.coalesce(
            F.col("paid_total"),
            F.lit(0).cast("decimal(12,2)")
        )
    )
    .withColumn(
        "difference",
        (
            F.col("order_total") -
            F.col("paid_total")
        ).cast("decimal(12,2)")
    )
    .withColumn(
        "payment_status",
        F.when(
            F.col("order_total").isNull(),
            "NO_ORDER_DETAILS"
        )
        .when(
            F.col("paid_total") == 0,
            "NO_PAYMENT"
        )
        .when(
            F.abs(F.col("difference").cast("double")) <= 0.01,
            "MATCH"
        )
        .when(
            F.col("paid_total") < F.col("order_total"),
            "UNDERPAID"
        )
        .otherwise("OVERPAID")
    )
)

print("\n=== Payment Status Distribution ===")

payment_analysis.groupBy("payment_status").count().orderBy(
    F.col("count").desc()
).show(truncate=False)


# ------------------------------------------------------------
# A3. How many orders are exactly matched?
# ------------------------------------------------------------

print("\n=== Exact Payment Matches ===")

payment_analysis.filter(
    F.col("payment_status") == "MATCH"
).select(
    "orderid",
    "order_total",
    "paid_total",
    "difference"
).show(20, truncate=False)


# ------------------------------------------------------------
# A4. Distribution of mismatch types
# ------------------------------------------------------------

print("\n=== Mismatch Type Distribution ===")

payment_analysis.filter(
    F.col("payment_status").isin(
        "UNDERPAID",
        "OVERPAID",
        "NO_PAYMENT"
    )
).groupBy(
    "payment_status"
).agg(
    F.count("*").alias("orders"),
    F.sum(
        F.abs(F.col("difference").cast("double"))
    ).alias("total_absolute_difference"),
    F.avg(
        F.abs(F.col("difference").cast("double"))
    ).alias("average_absolute_difference")
).show(truncate=False)


# ------------------------------------------------------------
# A5. Inspect payment records for a few mismatches
# ------------------------------------------------------------

print("\n=== Sample Mismatched Orders ===")

sample_mismatches = (
    payment_analysis
    .filter(
        F.col("payment_status").isin(
            "UNDERPAID",
            "OVERPAID"
        )
    )
    .orderBy(
        F.abs(F.col("difference").cast("double")).desc()
    )
    .limit(10)
)

sample_mismatches.show(truncate=False)


# ------------------------------------------------------------
# A6. Inspect the underlying payment rows
# ------------------------------------------------------------

sample_order_ids = [
    row["orderid"]
    for row in sample_mismatches.select("orderid").collect()
]

print("\n=== Underlying Order Details ===")

if sample_order_ids:
    order_details.filter(
        F.col("orderid").isin(sample_order_ids)
    ).orderBy("orderid").show(50, truncate=False)

print("\n=== Underlying Payments ===")

if sample_order_ids:
    payments.filter(
        F.col("orderid").isin(sample_order_ids)
    ).orderBy("orderid").show(50, truncate=False)


# ============================================================
# B. MISSING SHIPMENT INVESTIGATION
# ============================================================

print("\n" + "=" * 70)
print("MISSING SHIPMENT INVESTIGATION")
print("=" * 70)


# ------------------------------------------------------------
# B1. Shipment coverage
# ------------------------------------------------------------

total_orders = orders.select("orderid").distinct().count()
total_shipments = shipments.select("shipmentid").distinct().count()

orders_with_shipment = (
    orders
    .join(
        shipments.select("orderid").distinct(),
        on="orderid",
        how="inner"
    )
    .select("orderid")
    .distinct()
)

orders_without_shipment = (
    orders
    .join(
        shipments.select("orderid").distinct(),
        on="orderid",
        how="left_anti"
    )
)

print("\n=== Shipment Coverage ===")
print("Total orders:", total_orders)
print("Total shipments:", total_shipments)
print("Orders with shipment:", orders_with_shipment.count())
print("Orders without shipment:", orders_without_shipment.count())


# ------------------------------------------------------------
# B2. Shipment coverage by order status
# ------------------------------------------------------------
# This is important because an order being unshipped may be
# perfectly consistent with some statuses.
# ------------------------------------------------------------

print("\n=== Missing Shipments by Order Status ===")

missing_shipment_by_status = (
    orders_without_shipment
    .groupBy("status")
    .count()
    .orderBy(F.col("count").desc())
)

missing_shipment_by_status.show(truncate=False)


# ------------------------------------------------------------
# B3. Compare shipment coverage for every status
# ------------------------------------------------------------

print("\n=== Shipment Coverage by Status ===")

shipment_status_analysis = (
    orders
    .join(
        shipments.select("orderid").distinct()
        .withColumn("has_shipment", F.lit(1)),
        on="orderid",
        how="left"
    )
    .withColumn(
        "shipment_status",
        F.when(
            F.col("has_shipment") == 1,
            "HAS_SHIPMENT"
        ).otherwise("NO_SHIPMENT")
    )
    .groupBy("status", "shipment_status")
    .count()
    .orderBy("status", "shipment_status")
)

shipment_status_analysis.show(truncate=False)


# ------------------------------------------------------------
# B4. Missing shipments by year
# ------------------------------------------------------------

print("\n=== Missing Shipments by Year ===")

orders_without_shipment_by_year = (
    orders_without_shipment
    .groupBy("year")
    .count()
    .orderBy("year")
)

orders_without_shipment_by_year.show(truncate=False)


# ------------------------------------------------------------
# B5. Inspect sample orders without shipments
# ------------------------------------------------------------

print("\n=== Sample Orders Without Shipment ===")

orders_without_shipment.select(
    "orderid",
    "customerid",
    "orderdate",
    "status",
    "total_order_amount",
    "total_order_quantity",
    "order_value_category"
).orderBy("orderid").show(30, truncate=False)


# ------------------------------------------------------------
# B6. Check whether missing shipments correlate with
#     cancelled/processing orders
# ------------------------------------------------------------

print("\n=== Missing Shipment Status Percentages ===")

missing_shipment_percentage = (
    orders_without_shipment
    .groupBy("status")
    .agg(
        F.count("*").alias("missing_shipments")
    )
)

total_by_status = (
    orders
    .groupBy("status")
    .agg(
        F.count("*").alias("total_orders")
    )
)

(
    total_by_status
    .join(
        missing_shipment_percentage,
        on="status",
        how="left"
    )
    .fillna(0, subset=["missing_shipments"])
    .withColumn(
        "missing_shipment_percentage",
        F.round(
            F.col("missing_shipments") /
            F.col("total_orders") * 100,
            2
        )
    )
    .orderBy("status")
    .show(truncate=False)
)


print("\n" + "=" * 70)
print("INVESTIGATION COMPLETE")
print("=" * 70)

PAYMENT MISMATCH INVESTIGATION

=== Payment Coverage ===
Orders without any payment: 20333

=== Payment Status Distribution ===
+----------------+-----+
|payment_status  |count|
+----------------+-----+
|UNDERPAID       |22425|
|NO_PAYMENT      |17551|
|NO_ORDER_DETAILS|6864 |
|OVERPAID        |3160 |
+----------------+-----+


=== Exact Payment Matches ===
+-------+-----------+----------+----------+
|orderid|order_total|paid_total|difference|
+-------+-----------+----------+----------+
+-------+-----------+----------+----------+


=== Mismatch Type Distribution ===
+--------------+------+-------------------------+---------------------------+
|payment_status|orders|total_absolute_difference|average_absolute_difference|
+--------------+------+-------------------------+---------------------------+
|OVERPAID      |3160  |8666890.129999984        |2742.686749999995          |
|UNDERPAID     |22425 |4.070232038500019E8      |18150.42157636575          |
|NO_PAYMENT    |17551 |3.445365208799

In [15]:
# ============================================================
# PART 9 — DATE ANALYSIS
# ============================================================

from pyspark.sql import functions as F
import logging

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------

logger = logging.getLogger("Part9_DateAnalysis")
logger.setLevel(logging.INFO)

logger.info("Starting Part 9 — Date Analysis")


# ============================================================
# 1. DAILY SALES
# ============================================================
# Calculate total sales and number of orders for each day.
# ============================================================

daily_sales = (
    sales
    .filter(F.col("orderdate").isNotNull())
    .groupBy("orderdate")
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("orderid").alias("total_orders"),
        F.sum("quantity").alias("total_quantity")
    )
    .orderBy("orderdate")
)

print("=== Daily Sales ===")
daily_sales.show(20, truncate=False)


# ============================================================
# 2. MONTHLY SALES
# ============================================================
# Group by year + month.
#
# Year is included so that January 2024 and January 2025
# remain separate periods.
# ============================================================

monthly_sales = (
    sales
    .filter(F.col("orderdate").isNotNull())
    .groupBy(
        F.year("orderdate").alias("year"),
        F.month("orderdate").alias("month")
    )
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("orderid").alias("total_orders"),
        F.sum("quantity").alias("total_quantity")
    )
    .orderBy("year", "month")
)

print("=== Monthly Sales ===")
monthly_sales.show(50, truncate=False)


# ============================================================
# 3. QUARTERLY SALES
# ============================================================

quarterly_sales = (
    sales
    .filter(F.col("orderdate").isNotNull())
    .groupBy(
        F.year("orderdate").alias("year"),
        F.quarter("orderdate").alias("quarter")
    )
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("orderid").alias("total_orders"),
        F.sum("quantity").alias("total_quantity")
    )
    .orderBy("year", "quarter")
)

print("=== Quarterly Sales ===")
quarterly_sales.show(20, truncate=False)


# ============================================================
# 4. YEARLY SALES
# ============================================================

yearly_sales = (
    sales
    .filter(F.col("orderdate").isNotNull())
    .groupBy(
        F.year("orderdate").alias("year")
    )
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("orderid").alias("total_orders"),
        F.sum("quantity").alias("total_quantity")
    )
    .orderBy("year")
)

print("=== Yearly Sales ===")
yearly_sales.show(20, truncate=False)


# ============================================================
# 5. MONTH WITH HIGHEST SALES
# ============================================================
# Rank monthly periods by total sales.
#
# We use row_number because we need the single highest-sales
# month. If two months have the same value, year/month provide
# deterministic ordering.
# ============================================================

highest_sales_month = (
    monthly_sales
    .orderBy(
        F.col("total_sales").desc(),
        F.col("year").desc(),
        F.col("month").desc()
    )
    .limit(1)
)

print("=== Month With Highest Sales ===")
highest_sales_month.show(truncate=False)


# ============================================================
# 6. DAY WITH HIGHEST NUMBER OF ORDERS
# ============================================================
# First aggregate orders by date, then find the date with
# the highest number of distinct orders.
# ============================================================

daily_orders = (
    sales
    .filter(F.col("orderdate").isNotNull())
    .groupBy("orderdate")
    .agg(
        F.countDistinct("orderid").alias("total_orders")
    )
)

highest_order_day = (
    daily_orders
    .orderBy(
        F.col("total_orders").desc(),
        F.col("orderdate").asc()
    )
    .limit(1)
)

print("=== Day With Highest Number of Orders ===")
highest_order_day.show(truncate=False)


# ============================================================
# 7. AVERAGE ORDERS PER MONTH
# ============================================================
# First calculate the number of orders in every month.
# Then calculate the average monthly order count.
# ============================================================

average_orders_per_month = (
    monthly_sales
    .agg(
        F.avg("total_orders").alias("average_orders_per_month")
    )
)

print("=== Average Orders Per Month ===")
average_orders_per_month.show(truncate=False)


# ============================================================
# 8. STORE PART 9 RESULTS
# ============================================================

part9_results = {
    "daily_sales": daily_sales,
    "monthly_sales": monthly_sales,
    "quarterly_sales": quarterly_sales,
    "yearly_sales": yearly_sales,
    "highest_sales_month": highest_sales_month,
    "highest_order_day": highest_order_day,
    "average_orders_per_month": average_orders_per_month
}


# ============================================================
# VALIDATION
# ============================================================

print("\n=== Part 9 Validation ===")

for name, df in part9_results.items():
    print(f"{name}: {df.count()} rows")


logger.info("Part 9 completed successfully.")

=== Daily Sales ===
+----------+-----------+------------+--------------+
|orderdate |total_sales|total_orders|total_quantity|
+----------+-----------+------------+--------------+
|2024-01-01|797698.05  |43          |534           |
|2024-01-02|554259.51  |34          |372           |
|2024-01-03|956922.36  |44          |624           |
|2024-01-04|571969.71  |34          |386           |
|2024-01-05|1146885.03 |51          |745           |
|2024-01-06|936421.25  |47          |586           |
|2024-01-07|787117.35  |41          |512           |
|2024-01-08|1128089.92 |51          |722           |
|2024-01-09|847836.17  |44          |549           |
|2024-01-10|617058.64  |42          |417           |
|2024-01-11|741428.71  |36          |454           |
|2024-01-12|1245924.25 |53          |733           |
|2024-01-13|953265.45  |49          |604           |
|2024-01-14|685556.94  |45          |561           |
|2024-01-15|889861.19  |52          |641           |
|2024-01-16|926077.21  |41

In [18]:
# ============================================================
# TEST — Verify S3 write permission
# ============================================================

TEST_PATH = "s3://aws-ecommerce-s3/ecommerce/processed/_permission_test"

(
    tables["customers"]
    .limit(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(TEST_PATH)
)

print("SUCCESS: Glue can write to the processed S3 location.")
print(TEST_PATH)

SUCCESS: Glue can write to the processed S3 location.
s3://aws-ecommerce-s3/ecommerce/processed/_permission_test


In [19]:
# ============================================================
# PART 10 — WRITE PROCESSED DATA TO S3
# ============================================================

from pyspark.sql import functions as F
import logging

# ------------------------------------------------------------
# Logging configuration
# ------------------------------------------------------------
logger = logging.getLogger("Part10_Output")
logger.setLevel(logging.INFO)

# ------------------------------------------------------------
# Processed S3 base path
# ------------------------------------------------------------
PROCESSED_PATH = "s3://aws-ecommerce-s3/ecommerce/processed"

logger.info("Starting Part 10 — Writing processed data to S3")


# ============================================================
# 1. Prepare output datasets
# ============================================================

# Core cleaned tables
output_tables = {
    "customers": tables["customers"],
    "products": tables["products"],
    "categories": tables["categories"],
    "order_details": tables["order_details"],
    "payments": tables["payments"],
    "shipments": tables["shipments"]
}


# ============================================================
# 2. Write customers
# ============================================================

logger.info("Writing customers...")

(
    output_tables["customers"]
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{PROCESSED_PATH}/customers")
)


# ============================================================
# 3. Write products
# ============================================================

logger.info("Writing products...")

(
    output_tables["products"]
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{PROCESSED_PATH}/products")
)


# ============================================================
# 4. Write categories
# ============================================================

logger.info("Writing categories...")

(
    output_tables["categories"]
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{PROCESSED_PATH}/categories")
)


# ============================================================
# 5. Write orders
#
# Orders must be partitioned by year and month.
# These columns were created in Part 3.
# ============================================================

logger.info("Writing orders partitioned by year/month...")

(
    tables["orders"]
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .partitionBy("year", "month")
    .save(f"{PROCESSED_PATH}/orders")
)


# ============================================================
# 6. Write order_details
# ============================================================

logger.info("Writing order_details...")

(
    output_tables["order_details"]
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{PROCESSED_PATH}/order_details")
)


# ============================================================
# 7. Write payments
# ============================================================

logger.info("Writing payments...")

(
    output_tables["payments"]
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{PROCESSED_PATH}/payments")
)


# ============================================================
# 8. Write shipments
# ============================================================

logger.info("Writing shipments...")

(
    output_tables["shipments"]
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{PROCESSED_PATH}/shipments")
)

# ============================================================
# 9. Write fact_sales
#
# fact_sales was created from:
# orders + order_details + products + categories
#
# Required columns:
# orderid
# customerid
# productid
# categoryid
# orderdate
# quantity
# unitprice
# total_amount
# status
# ============================================================

logger.info("Writing fact_sales...")

fact_sales = (
    sales
    .select(
        "orderid",
        "customerid",
        "productid",
        "categoryid",
        "orderdate",
        "quantity",
        "unitprice",
        "total_amount",
        "status"
    )
)

(
    fact_sales
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{PROCESSED_PATH}/fact_sales")
)


# ============================================================
# 10. Write customer_sales
# ============================================================

logger.info("Writing customer_sales...")

(
    customer_sales
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{PROCESSED_PATH}/customer_sales")
)


# ============================================================
# 11. Write product_sales
# ============================================================

logger.info("Writing product_sales...")

(
    product_sales
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{PROCESSED_PATH}/product_sales")
)


# ============================================================
# 12. Write category_sales
# ============================================================

logger.info("Writing category_sales...")

(
    category_sales
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{PROCESSED_PATH}/category_sales")
)


# ============================================================
# 13. Write monthly_sales
# ============================================================

logger.info("Writing monthly_sales...")

(
    monthly_sales
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{PROCESSED_PATH}/monthly_sales")
)


# ============================================================
# 14. Validation
# ============================================================

print("\n" + "=" * 60)
print("PART 10 — OUTPUT VALIDATION")
print("=" * 60)

output_paths = {
    "customers": f"{PROCESSED_PATH}/customers",
    "products": f"{PROCESSED_PATH}/products",
    "categories": f"{PROCESSED_PATH}/categories",
    "orders": f"{PROCESSED_PATH}/orders",
    "order_details": f"{PROCESSED_PATH}/order_details",
    "payments": f"{PROCESSED_PATH}/payments",
    "shipments": f"{PROCESSED_PATH}/shipments",
    "fact_sales": f"{PROCESSED_PATH}/fact_sales",
    "customer_sales": f"{PROCESSED_PATH}/customer_sales",
    "product_sales": f"{PROCESSED_PATH}/product_sales",
    "category_sales": f"{PROCESSED_PATH}/category_sales",
    "monthly_sales": f"{PROCESSED_PATH}/monthly_sales"
}

for name, path in output_paths.items():
    print(f"{name:<20} -> {path}")

print("\nOutput format : Parquet")
print("Compression    : Snappy")
print("Orders         : Partitioned by year/month")

logger.info("Part 10 completed successfully.")


PART 10 — OUTPUT VALIDATION
customers            -> s3://aws-ecommerce-s3/ecommerce/processed/customers
products             -> s3://aws-ecommerce-s3/ecommerce/processed/products
categories           -> s3://aws-ecommerce-s3/ecommerce/processed/categories
orders               -> s3://aws-ecommerce-s3/ecommerce/processed/orders
order_details        -> s3://aws-ecommerce-s3/ecommerce/processed/order_details
payments             -> s3://aws-ecommerce-s3/ecommerce/processed/payments
shipments            -> s3://aws-ecommerce-s3/ecommerce/processed/shipments
fact_sales           -> s3://aws-ecommerce-s3/ecommerce/processed/fact_sales
customer_sales       -> s3://aws-ecommerce-s3/ecommerce/processed/customer_sales
product_sales        -> s3://aws-ecommerce-s3/ecommerce/processed/product_sales
category_sales       -> s3://aws-ecommerce-s3/ecommerce/processed/category_sales
monthly_sales        -> s3://aws-ecommerce-s3/ecommerce/processed/monthly_sales

Output format : Parquet
Compression    

In [21]:
# ============================================================
# PART 11 — FINAL ANALYTICAL DATA MODEL
# ============================================================
# Purpose:
#   Build the final analytical model and save it to:
#
#   s3://aws-ecommerce-s3/ecommerce/processed/FinalDataModel/
#
# Output datasets:
#   1. fact_sales
#   2. customer_sales
#   3. product_sales
#   4. category_sales
#   5. monthly_sales
#
# Format:
#   Parquet + Snappy
#
# The outputs are designed to be discovered later by
# an AWS Glue Crawler and queried through Athena.
# ============================================================

from pyspark.sql import functions as F
import logging

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------

logger = logging.getLogger("Part11_FinalDataModel")
logger.setLevel(logging.INFO)

logger.info("Starting Part 11 — Final Analytical Data Model")


# ------------------------------------------------------------
# Output path
# ------------------------------------------------------------

FINAL_MODEL_PATH = (
    "s3://aws-ecommerce-s3/ecommerce/processed/FinalDataModel"
)

logger.info(f"Final model output path: {FINAL_MODEL_PATH}")


# ============================================================
# 1. FACT SALES
# ============================================================
# Grain:
#   One row represents one order-detail/product line.
#
# Required columns:
#   orderid
#   customerid
#   productid
#   categoryid
#   orderdate
#   quantity
#   unitprice
#   total_amount
#   status
# ============================================================

logger.info("Building fact_sales...")

fact_sales = (
    sales
    .select(
        "orderid",
        "customerid",
        "productid",
        "categoryid",
        "orderdate",
        "quantity",
        "unitprice",
        "total_amount",
        "status"
    )
    .dropDuplicates()
)

logger.info(
    f"fact_sales created with {fact_sales.count()} rows"
)


# ============================================================
# 2. ORDER-LEVEL TOTALS
# ============================================================
# This intermediate dataset is used to correctly calculate
# average order value at customer level.
#
# We do NOT calculate:
#
#     AVG(line total)
#
# because an order may contain multiple products/lines.
#
# Instead:
#
#     1. Calculate total value of each order
#     2. Calculate average of those order totals
# ============================================================

logger.info("Calculating order-level totals...")

order_totals = (
    fact_sales
    .groupBy(
        "orderid",
        "customerid"
    )
    .agg(
        F.sum("total_amount").alias("order_total")
    )
)


# ============================================================
# 3. CUSTOMER SALES
# ============================================================
# One row per customer.
#
# Metrics:
#   total_sales
#   total_orders
#   total_quantity
#   average_order_value
# ============================================================

logger.info("Building customer_sales...")

customer_sales = (
    fact_sales
    .groupBy("customerid")
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("orderid").alias("total_orders"),
        F.sum("quantity").alias("total_quantity")
    )
    .join(
        order_totals
        .groupBy("customerid")
        .agg(
            F.avg("order_total").alias("average_order_value")
        ),
        on="customerid",
        how="left"
    )
    .select(
        "customerid",
        "total_sales",
        "total_orders",
        "total_quantity",
        "average_order_value"
    )
)


# ============================================================
# 4. PRODUCT SALES
# ============================================================
# One row per product.
#
# Metrics:
#   total_sales
#   total_quantity
#   total_orders
# ============================================================

logger.info("Building product_sales...")

product_sales = (
    fact_sales
    .groupBy("productid")
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("orderid").alias("total_orders")
    )
)


# ============================================================
# 5. CATEGORY SALES
# ============================================================
# One row per category.
#
# Metrics:
#   total_sales
#   total_quantity
#   total_orders
# ============================================================

logger.info("Building category_sales...")

category_sales = (
    fact_sales
    .groupBy("categoryid")
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("orderid").alias("total_orders")
    )
)


# ============================================================
# 6. MONTHLY SALES
# ============================================================
# One row per year/month.
#
# Metrics:
#   total_sales
#   total_orders
#   total_quantity
# ============================================================

logger.info("Building monthly_sales...")

monthly_sales = (
    fact_sales
    .filter(F.col("orderdate").isNotNull())
    .groupBy(
        F.year("orderdate").alias("year"),
        F.month("orderdate").alias("month")
    )
    .agg(
        F.sum("total_amount").alias("total_sales"),
        F.countDistinct("orderid").alias("total_orders"),
        F.sum("quantity").alias("total_quantity")
    )
    .orderBy("year", "month")
)


# ============================================================
# 7. WRITE FINAL MODEL TO S3
# ============================================================
# Same format as Part 10:
#
#   Format      : Parquet
#   Compression : Snappy
# ============================================================

logger.info("Writing final analytical model to S3...")

(
    fact_sales
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{FINAL_MODEL_PATH}/fact_sales")
)

(
    customer_sales
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{FINAL_MODEL_PATH}/customer_sales")
)

(
    product_sales
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{FINAL_MODEL_PATH}/product_sales")
)

(
    category_sales
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{FINAL_MODEL_PATH}/category_sales")
)

(
    monthly_sales
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(f"{FINAL_MODEL_PATH}/monthly_sales")
)


# ============================================================
# 8. VALIDATION
# ============================================================

final_model = {
    "fact_sales": fact_sales,
    "customer_sales": customer_sales,
    "product_sales": product_sales,
    "category_sales": category_sales,
    "monthly_sales": monthly_sales
}

print("\n" + "=" * 70)
print("PART 11 — FINAL DATA MODEL VALIDATION")
print("=" * 70)

for name, df in final_model.items():

    print(f"\n{name}")
    print("-" * 70)

    print(f"Rows    : {df.count()}")
    print(f"Columns : {len(df.columns)}")
    print(f"Path    : {FINAL_MODEL_PATH}/{name}")

    print("\nSchema:")
    df.printSchema()


print("\n" + "=" * 70)
print("FINAL MODEL OUTPUT")
print("=" * 70)

print(f"Base path   : {FINAL_MODEL_PATH}")
print("Format      : Parquet")
print("Compression : Snappy")

print("\nDatasets:")
for name in final_model.keys():
    print(f"  - {name}")

logger.info("Part 11 completed successfully.")


PART 11 — FINAL DATA MODEL VALIDATION

fact_sales
----------------------------------------------------------------------
Rows    : 99987
Columns : 9
Path    : s3://aws-ecommerce-s3/ecommerce/processed/FinalDataModel/fact_sales

Schema:
root
 |-- orderid: long (nullable = true)
 |-- customerid: long (nullable = true)
 |-- productid: long (nullable = true)
 |-- categoryid: long (nullable = true)
 |-- orderdate: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unitprice: decimal(12,2) (nullable = true)
 |-- total_amount: decimal(12,2) (nullable = true)
 |-- status: string (nullable = true)


customer_sales
----------------------------------------------------------------------
Rows    : 9859
Columns : 5
Path    : s3://aws-ecommerce-s3/ecommerce/processed/FinalDataModel/customer_sales

Schema:
root
 |-- customerid: long (nullable = true)
 |-- total_sales: decimal(22,2) (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_quantity: long (nullable = tru